In [1]:
import os
import cv2
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog



In [2]:

def extract_color_hog_features(img, pixels_per_cell=(8, 8), cells_per_block=(2, 2)):
    features = []
    for channel in cv2.split(img):  # Split into R, G, B channels
        features_channel, _ = hog(channel, pixels_per_cell=pixels_per_cell, 
                               cells_per_block=cells_per_block, visualize=True)
        features.append(features_channel)
    
    return np.concatenate(features)  # Concatenate features from all channels


In [3]:
# Load and preprocess the data
data_path = 'C://Users//Sinchan A//Desktop//Internship//vid'
images = []
labels = []

for folder in ['fake','real']:
    folder_path = os.path.join(data_path, folder)
    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (128, 128))  # Resize for faster processing
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
        images.append(img)
        labels.append(0 if folder == 'real' else 1)  # 0 for real, 1 for fake


In [4]:
# Extract HOG features
features = [extract_color_hog_features(img) for img in images]


In [5]:
# Convert lists to numpy arrays
features = np.array(features)
labels = np.array(labels)

In [6]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.4, random_state=42)

In [7]:
from imblearn.over_sampling import SMOTE
from collections import Counter

counter = Counter(y_train)
print('Before', counter)

# oversampling the train dataset using SMOTE
smt = SMOTE()
X_train, y_train = smt.fit_resample(X_train, y_train)

counter = Counter(y_train)
print('After', counter)

Before Counter({0: 2805, 1: 2565})
After Counter({0: 2805, 1: 2805})


In [8]:
# Scale the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
# Create and train the Random Forest classifier
rf = RandomForestClassifier(
    n_estimators=100,  # Number of trees
    max_depth=None,  # No depth limit
    min_samples_split=2,  # Minimum number of samples to split a node
    min_samples_leaf=1,  # Minimum number of samples in a leaf node
    max_features='sqrt',  # Use sqrt(n_features) features for each tree
    bootstrap=True,  # Use bootstrap samples
    n_jobs=-1,  # Use all CPU cores
    random_state=42
)

In [10]:
rf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf.predict(X_test)

In [11]:
# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)




In [12]:
print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1-score: {f1}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1823
           1       1.00      1.00      1.00      1758

    accuracy                           1.00      3581
   macro avg       1.00      1.00      1.00      3581
weighted avg       1.00      1.00      1.00      3581

